# **Lab 3: Accelerating ExecuTorch on Arm Ethos-U Neural Processors**
### NPU Inference on Corstone-320 Fixed Virtual Platform (Ethos-U85), TOSA, and Google's Model Explorer

## **Introduction**

Welcome to the **Accelerating ExecuTorch on Arm Neural Processors** lab! In this hands-on session, you will progress from considering Arm-powered CPUs, to deploying on the Ethos-U Neural Processing Unit backend. You will also learn about the Tensor Operator Set Architecture (TOSA) intermediate representation (IR), and how you can utilise tools such as Google's Model Explorer, via adapters developed by Arm, to analyse models before deploying to different backends.

You will understand why you might consider using an Ethos-U NPU over a CPU, and how you delegate to this backend whilst still performing lowering and quantization as before. The lab will showcase the streamlined way to simulate hardware using Arm's Fixed Virtual Platforms (FVP), including pre-existing workflow scripts and launchpads you can utilise to speed-up prototyping.

**Requirements**: To complete this lab, you are recommended to utilise a Raspberry Pi 5 - or similar Arm-powered Edge device - running a Linux-based OS.

### **Lab Objectives:** 

The objectives of this lab are as follows:

1. **Understand Neural Processing Units (NPU)**:
   - a

2. **Understand the Tensor Operator Set Architecture (TOSA)**:
   - a

3. **Learn how to utilise tools to assess models**:
   - a

4. **Identify and use workflow scripts for deploying a simple application on an Ethos-U FVP**:
   - a

### **Prerequisites**

To follow this lab, you should have a basic understanding of:

- **Python Programming**: Experience writing and running Python scripts.
- **AI/ML principles**: Basic understanding of AI/ML and model types.

> Note: It is assumed that the same Jupyter Kernel, is used throughout this lab and that all cells are run sequentially

### **Recommended**
- It is recommended to complete Labs 1 and 2 prior to this lab.
- Non-essential but recommended prior to this lab is to complete the [Optimising GenAI on Arm](https://www.edx.org/learn/computer-science/arm-education-ai-on-arm) course at Edx or Coursera.


## **Pre-lab set-up**

```bash
# 1. Make sure you have run the initial_setup script (should have been run if you have previously completed lab 1 or 2) (this installs Python 3.11, creates a virtual env, and sets up all required libraries). Select the NPU_lab_venv as the Jupyter kernel.
bash setup.sh

# 2. Run the NPU_venv_setup script to install additional dependencies required for this lab.
```


In [16]:
%pip install ai-edge-model-explorer pte-adapter-model-explorer

  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 MB 9.7 MB/s  0:00:08m0:00:0100:01m
Using cached flatbuffers-25.12.19-py2.py3-none-any.whl (26 kB)
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 24.3.25
    Uninstalling flatbuffers-24.3.25:
      Successfully uninstalled flatbuffers-24.3.25
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [vgf-adapter-model-explorer]er-model-explorer]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tosa-tools 2025.11.0 requires jsonschema==4.24.0, which is not installed.
tosa-tools 2025.11.0 requires semver==3.0.4, which is not installed.
ethos-u-vela 4.5.0 requires flatbuffers==24.3.25, but you have flatbuffers 25.12.19 which is incompatible.
tosa-tools 2025.11.0 requires flatbuffers==25.2.10, but you have flatbuffer


## **The Tensor Operator Set Architecture (TOSA)**

### **What is TOSA?**
The **Tensor Operator Set Architecture (TOSA)** is a minimal and stable set of tensor-level operators designed to **standardize machine learning model execution across hardware platforms**. [TOSA](https://www.mlplatform.org/tosa/) is framework-agnostic and includes precise functional and numerical definitions to ensure consistent results across CPUs, GPUs, and NPUs. The latest specification can be found [here](https://www.mlplatform.org/tosa/tosa_spec.html), and is provided as a [GitHub repository](https://github.com/arm/tosa-specification) as well.

### **Why Do We Need TOSA?**
Modern ML frameworks such as [Executorch](https://github.com/pytorch/executorch), [LiteRT](https://ai.google.dev/edge/litert), and [JAX](https://github.com/jax-ml/jax) define hundreds (sometimes thousands) of distinct operators.  This diversity creates challenges when deploying models across platforms or converting them between frameworks (e.g., PyTorch → TensorFlow conversion is currently non-trivial due to different tensor layout conventions). 

**TOSA solves this problem** by providing a common set of standard operators that act as a stable intermediate representation (IR), a kind of “middle language” between source code and machine instructions that can be further compiled to a diverse range of backend targets. You can think of it as a bit like an assembly language. Subtle variations in the way operators are implemented in different frameworks, such as `CONV2D` for [2D convolutions](https://www.mlplatform.org/tosa/tosa_spec_1_0_1.html#_conv2d) are specified in detail so that ML developers a model will perform will perform as expected across devices and frameworks with an acceptably low level of rounding errors. 

### **TOSA in the ExecuTorch Compilation Flow**

TOSA was added to PyTorch and ExecuTorch through [collaboration between Arm and Meta](https://developer.arm.com/community/arm-community-blogs/b/ai-blog/posts/executorch-and-tosa-enabling-pytorch-on-arm-platforms) in 2023.

**So, why did we not use TOSA in Labs 1 and 2?**

In the earlier XNNPACK CPU labs, TOSA was not used. Instead, ExecuTorch lowered models directly to the XNNPACK backend, which is a highly optimized CPU kernel library. Because the target was a general-purpose CPU with an established execution provider, there was no need for an additional intermediate representation. The flow was simpler: PyTorch → XNNPACK, avoiding the extra PyTorch → TOSA → backend step.

In principle, we could have lowered to TOSA first, but there would have been little practical benefit. XNNPACK is already a mature, CPU-optimized kernel library with its own well-defined operator expectations, and ExecuTorch can lower PyTorch operators directly to it. TOSA is most valuable when creating a stable contract between frameworks and new systems, particularly those that are heterogeneous (e.g., systems that include NPUs/AI accelerators or GPUs). Instead of implementing hundreds of framework-specific operators for each new device, a vendor can implement the smaller, stable TOSA operator set and rely on ExecuTorch to lower models into that form.

Since this lab will extend from CPU inference to consider the Ethos-U NPU, it is an appropriate time to introduce TOSA

### **Key Benefits of TOSA**
- **Standardization**: A consistent, well-defined target for both ML frameworks and hardware vendors.  
- **Portability**: TOSA-compliant models run on any hardware that supports TOSA, with guaranteed numerical consistency.  
- **Future-Proofing**: Future novel operators not directly in TOSA can still be expressed using existing TOSA primitives, ensuring long-term flexibility.  

## **Assessing TOSA Compatability**

We are going to examine models in their TOSA representation using Google's Model Explorer

In the previous lab, we utilised MobileNetV2, a CNN designed for mobile and embedded devices.Before resuming with MobileNetV2, we will briefly utilise ResNet18. ResNet18 is a classic convolutional neural network introduced in 2015 that uses residual (skip) connections to make deeper networks easier to train. It has 18 layers, around 11.7 million parameters, and uses standard 3×3 convolutions throughout. Because of its simple and regular structure, it is very stable for export, quantization, and backend lowering, which makes it a strong baseline model in deployment workflows.

MobileNetV2, by contrast, replaces standard convolutions with depthwise separable convolutions and uses inverted residual blocks with linear bottlenecks to drastically reduce computation and model size (~3.4 million parameters). This makes MobileNetV2 much more efficient for NPUs and edge devices, but its more complex operator patterns can make it slightly more sensitive during quantization and backend lowering.

The goal is to produce a portable ExecuTorch artifact (.pte) that we can inspect and compare with later lowered/partitioned outputs.

In [2]:
import os
import torch
from torchvision import models

SAVE_DIR = "executorch_demo"
os.makedirs(SAVE_DIR, exist_ok=True)

def to_executorch(pt_model: torch.nn.Module, example_inputs, out_path: str):
    pt_model.eval()

    exported = torch.export.export(pt_model, example_inputs)

    from executorch.exir import to_edge
    edge_mgr = to_edge(exported)

    et_mgr = edge_mgr.to_executorch()

    
    with open(out_path, "wb") as f:
        et_mgr.write_to_file(f)

    return out_path


model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
example = (torch.randn(1, 3, 224, 224),)

out_path = os.path.join(SAVE_DIR, "resnet18.pte")
to_executorch(model, example, out_path)
print(os.path.getsize(out_path), "bytes")
out_path


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/ubuntu/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 32.6MB/s]
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)


46825648 bytes


'executorch_demo/resnet18.pte'

## **What are Neural Processing Units? What is the Ethos-U?**

- NPU
- Ethos-U, U55, 65, 85, configurations of MAC units, pairing with Cortex-A and M

## **Building a non-TOSA-compatible MobileNetV2**

In this contrived example, we will construct a MobileNetV2-based model that intentionally breaks TOSA compatibility by inserting an LRN (Local Response Normalization) operation. This operation was proposed in the original AlexNet paper from 2012 and is not part of the TOSA specification. 

**What below cell does**
- Creates a backbone `MobileNetV2` 
- Inserts `tf.nn.local_response_normalization` (`LRN`) that is **not supported by TOSA**.
- Adds GAP + 1000-unit Dense “logits” layer and copies weights from a reference MobileNetV2 with top to preserve ImageNet logits.
- Exports a SavedModel and converts it to `.pte`.

In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights


SAVE_DIR = "executorch_demo"
os.makedirs(SAVE_DIR, exist_ok=True)

class MobileNetV2_LRN_Break(nn.Module):
    """
    PyTorch equivalent of:
      MobileNetV2(include_top=False) -> LRN -> GAP -> Dense(1000)
    with Dense weights copied from pretrained MobileNetV2(include_top=True).
    """
    def __init__(self, backbone_weights="imagenet"):
        super().__init__()

        # Reference model WITH top to copy 1k linear weights
        if backbone_weights == "imagenet":
            ref = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
            feat = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
        else:
            ref = mobilenet_v2(weights=None)
            feat = mobilenet_v2(weights=None)

        # Backbone without top: torchvision splits into features + classifier
        self.features = feat.features  # outputs [N, 1280, 7, 7] for 224x224

        # LRN: TF depth_radius=2 => PyTorch LocalResponseNorm size=2*2+1=5
        # TF params: bias=1.0, alpha=1e-4, beta=0.75
        self.lrn = nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=1.0)

        # GAP + logits
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.logits = nn.Linear(1280, 1000)

        # Copy pretrained classifier weights (the Linear inside ref.classifier)
        # torchvision MobileNetV2 classifier is: Sequential(Dropout, Linear)
        ref_linear = None
        for m in ref.classifier.modules():
            if isinstance(m, nn.Linear) and m.out_features == 1000:
                ref_linear = m
                break
        if ref_linear is None:
            raise RuntimeError("Could not find 1000-way Linear layer in reference MobileNetV2.")

        with torch.no_grad():
            self.logits.weight.copy_(ref_linear.weight)
            self.logits.bias.copy_(ref_linear.bias)

    def forward(self, x):
        x = self.features(x)      # [N, C, H, W]
        x = self.lrn(x)           # LRN "break"
        x = self.pool(x)          # [N, C, 1, 1]
        x = torch.flatten(x, 1)   # [N, C]
        x = self.logits(x)        # [N, 1000] (logits)
        return x

# -------------------------
# ExecuTorch export 
# -------------------------
def to_executorch(pt_model: torch.nn.Module, example_inputs, out_path: str):
    pt_model.eval()
    exported = torch.export.export(pt_model, example_inputs)

    from executorch.exir import to_edge
    edge_mgr = to_edge(exported)
    et_mgr = edge_mgr.to_executorch()

    with open(out_path, "wb") as f:
        et_mgr.write_to_file(f)

    return out_path

# Build model (equivalent to mobilenetv2_with_lrn_break())
m_break = MobileNetV2_LRN_Break(backbone_weights="imagenet")

# Export to ExecuTorch .pte 
example = (torch.randn(1, 3, 224, 224),)
out_path = os.path.join(SAVE_DIR, "mnetv2_lrn_break.pte")
to_executorch(m_break, example, out_path)

out_path

# -------------------------
# ADDENDUM: Ethos-U quantize + lower to produce an FVP-ready .pte
# (Corstone-320 / Ethos-U85)
# -------------------------
import copy
import torch

def to_executorch_ethosu_fvp(
    pt_model: torch.nn.Module,
    example_inputs,
    out_path: str,
    *,
    target: str = "ethos-u85-128",
    system_config: str = "Ethos_U85_SYS_DRAM_Mid",
    memory_mode: str = "Shared_Sram",
    extra_flags: str | None = None,
):
    """
    Create a .pte suitable for the Corstone-320 FVP Ethos-U flow:
      - PT2E post-training quantization (int8)
      - Partition + lower to Ethos-U backend (TOSA + Vela compile)
      - Serialize .pte

    Notes:
    """
    pt_model.eval()

    # Ethos-U backend components
    from executorch.backends.arm.ethosu import EthosUPartitioner
    from executorch.backends.arm.quantizer.arm_quantizer import (
        EthosUQuantizer,
        get_symmetric_quantization_config,
    )
    from executorch.exir import (
        EdgeCompileConfig,
        ExecutorchBackendConfig,
        to_edge_transform_and_lower,
    )
    from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e

    # ---- Compile spec ----
    from executorch.backends.arm.ethosu import EthosUCompileSpec  # type: ignore

    compile_spec = EthosUCompileSpec(
            target=target,
            system_config=system_config,
            memory_mode=memory_mode,
            extra_flags=extra_flags or "",
        )

    # ---- PT2E post-training quantization (EthosUQuantizer) ----
    graph_module = torch.export.export(pt_model, example_inputs).module()

    quantizer = EthosUQuantizer(compile_spec)
    qconfig = get_symmetric_quantization_config(is_per_channel=False)
    quantizer.set_global(qconfig)

    graph_module = prepare_pt2e(graph_module, quantizer)
    graph_module(*example_inputs)  # calibration run (simple representative pass)
    graph_module = convert_pt2e(graph_module)

    quant_exported_program = torch.export.export(graph_module, example_inputs)

    # ---- Lower/partition to Ethos-U and write .pte ----
    edge_pm = to_edge_transform_and_lower(
        quant_exported_program,
        partitioner=[EthosUPartitioner(compile_spec)],
        compile_config=EdgeCompileConfig(_check_ir_validity=False),
    ).to_executorch(
        config=ExecutorchBackendConfig(extract_delegate_segments=False)
    )

    with open(out_path, "wb") as f:
        edge_pm.write_to_file(f)

    return out_path


# -------------------------
# Use it for your MobileNetV2_LRN_Break
# -------------------------
example = (torch.randn(1, 3, 224, 224),)

# LRN is commonly the thing that prevents a "fully integer" Ethos-U export.
# For the FVP/Ethos-U deploy build, bypass it.
m_fvp = copy.deepcopy(m_break)
m_fvp.lrn = torch.nn.Identity()

out_path = os.path.join(SAVE_DIR, "mnetv2_fvp_ethosu85_128.pte")
to_executorch_ethosu_fvp(
    m_fvp,
    example,
    out_path,
    target="ethos-u85-128",                 # must match your FVP num_macs
    system_config="Ethos_U85_SYS_DRAM_Mid", # common Corstone-320 DRAM config
    memory_mode="Shared_Sram",
)

print("Wrote:", out_path, "bytes:", os.path.getsize(out_path))


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /home/ubuntu/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 16.2MB/s]
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)


ExportPassBaseError: DecomposeSelectScatterPass: call_module is not supported.

While executing %_guards_fn : [num_users=0] = call_module[target=_guards_fn](args = (%x,), kwargs = {})
Original traceback:
None
Use tlparse to see full graph. (https://github.com/pytorch/tlparse?tab=readme-ov-file#tlparse-parse-structured-pt2-logs)

Add some words about lowering via the edge and transform function etc please

Lower to tosa of our above resnet model




---

## Additional Tools and Models Used in This Lab

- **[MobileNetV2 Model](https://huggingface.co/docs/transformers/en/model_doc/mobilenet_v2)**:  
  A convolutional neural network (CNN) architecture introduced by Google (2018), designed for mobile and edge devices where compute and memory are limited.  

- **Model Explorer**:  
  An Arm-built visualization tool that provides an **intuitive, hierarchical view** of model graphs.  
  It organizes model operations into nested layers, allowing users to dynamically expand or collapse them for easier analysis.
  Repository: [Model Explorer](https://model-explorer.staging.tools.arm.com) 



Now we will look at lowering our above resnet model we made right at the start that doesn't break the rules of tosa, into its tosa buffer to be read from a folder 

In [5]:
from pathlib import Path
from torch.export import export
from executorch.exir import to_edge_transform_and_lower, EdgeCompileConfig
from executorch.backends.arm.tosa import TosaSpecification
from executorch.backends.arm.tosa.compile_spec import TosaCompileSpec
from executorch.backends.arm.util._factory import create_partitioner

model = model.eval()
ep = export(model,example) 

out_dir = Path("Executorch_intermdiates")
out_dir.mkdir(parents= True, exist_ok= True)

# 2) Create a TOSA compile spec (choose FP or INT profile)
tosa_spec = TosaSpecification.create_from_string("TOSA-1.0+FP")
compile_spec = TosaCompileSpec(tosa_spec)

# 3) Tell the backend to dump intermediates (this is where the .tosa appears)
out_dir = Path("tosa_dump")
out_dir.mkdir(parents=True, exist_ok=True)
compile_spec.dump_intermediate_artifacts_to(str(out_dir))

# 4) Partition + lower 
partitioner = create_partitioner(compile_spec)
edge_mgr = to_edge_transform_and_lower(
    ep,
    partitioner=[partitioner],
    compile_config=EdgeCompileConfig(_check_ir_validity=False),
)

print("Done. Look for a .tosa file under:", out_dir)



/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


Done. Look for a .tosa file under: tosa_dump


Here is where we use the model explorer to visualise the model, please select the file in the tosa_dump folder and then select model.pte or the example resnet.pte in our demo folder. 

The model explorer will let you visualise the models operations and their order. its best usages are in identify how the operations, their ordering and the inner workings of the model are different between formats in this lab we will ask yu to look at the difference ebtween .pte or the base executorch program and the .tosa format which is the series of buffers inside a model 


In [17]:
%%bash

model-explorer --extensions=pte_adapter_model_explorer

Loading extensions...

Loaded 11 adapters:
 - TFLite adapter (Flatbuffer)
 - TFLite adapter (MLIR)
 - TF adapter (MLIR)
 - TF adapter (direct)
 - GraphDef adapter
 - Pytorch adapter (exported program)
 - MLIR adapter
 - TOSA Flatbuffer Adapter
 - VGF Adapter
 - PTE Adapter
 - JSON adapter

Starting Model Explorer server at:
http://localhost:8080

Press Ctrl+C to stop.
Stopping server...


TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType

Copy this into your terminal to run the model explorer without cluttering the notebook: 



model-explorer --extension=pte_adapter_model_explorer


you can use this to visualise your .pte model in the base folder and your .tosa file found in the the above mentioned folder

there is an extension for .vgf and this will come in later labs

And here is where we can execute a set of commands to use the FVP. 

In [1]:
%%bash 

pwd

source ../executorch/examples/arm/arm-scratch/setup_path.sh

/home/ubuntu/learn_executorch_on_arm


That is the command to add the FVP's to PATH

In [ ]:
%%bash

source ../executorch/examples/arm/arm-scratch/setup_path.sh

source ../executorch/examples/arm/run.sh \
--aot_arm_compiler_flags="--delegate --quantize --intermediates mv2_u85/ --debug --evaluate" \
--output=mv2_u85 \
--target=ethos-u85-128 \
--model_name=mv2

+ cd /home/ubuntu/executorch
+ set +x
+ cmake_args=(-DCMAKE_TOOLCHAIN_FILE=${toolchain_cmake} -DCMAKE_BUILD_TYPE=${build_type} -DEXECUTORCH_BUILD_DEVTOOLS=${build_devtools} -DEXECUTORCH_BUILD_ARM_ETDUMP=${build_with_etdump})
+ [[ 0 -eq 1 ]]


--------------------------------------------------------------------------------
Build ExecuTorch target libs Release into '/home/ubuntu/executorch/arm_test/cmake-out'
--------------------------------------------------------------------------------


+ cmake -S /home/ubuntu/executorch -B /home/ubuntu/executorch/arm_test/cmake-out -DCMAKE_TOOLCHAIN_FILE=/home/ubuntu/executorch/examples/arm/ethos-u-setup/arm-none-eabi-gcc.cmake -DCMAKE_BUILD_TYPE=Release -DEXECUTORCH_BUILD_DEVTOOLS=OFF -DEXECUTORCH_BUILD_ARM_ETDUMP=OFF --preset arm-baremetal


Preset CMake variables:

  EXECUTORCH_BUILD_PRESET_FILE="/home/ubuntu/executorch/tools/cmake/preset/arm_baremetal.cmake"

-- The C compiler identification is GNU 13.3.1
-- The CXX compiler identification is GNU 13.3.1
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /home/ubuntu/executorch/examples/arm/arm-scratch/arm-gnu-toolchain-13.3.rel1-aarch64-arm-none-eabi/bin/arm-none-eabi-gcc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /home/ubuntu/executorch/examples/arm/arm-scratch/arm-gnu-toolchain-13.3.rel1-aarch64-arm-none-eabi/bin/arm-none-eabi-g++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- ExecuTorch version: 1.2.0
-- Found Python3: /home/ubuntu/.pyenv/versions/3.11.14/envs/aws_venv_3.11.14/bin/python3.11 (found version "3.11.14") found com

CMake Deprecation Warning at third-party/gflags/CMakeLists.txt:73 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.




...

/home/ubuntu/executorch/examples/arm/arm-scratch/arm-gnu-toolchain-13.3.rel1-aarch64-arm-none-eabi/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: warning: section `.ddr' type changed to PROGBITS
/home/ubuntu/executorch/examples/arm/arm-scratch/arm-gnu-toolchain-13.3.rel1-aarch64-arm-none-eabi/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: arm_executor_runner: section `.sram' can't be allocated in segment 2
LOAD: .ddr .sram
/home/ubuntu/executorch/examples/arm/arm-scratch/arm-gnu-toolchain-13.3.rel1-aarch64-arm-none-eabi/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: warning: arm_executor_runner has a LOAD segment with RWX permissions


[100%] Built target arm_executor_runner
[backends/arm/scripts/build_executor_runner.sh] Generated arm-none-eabi-gcc elf file:
/home/ubuntu/executorch/mv2_u85/mv2_arm_delegate_ethos-u85-128/cmake-out/arm_executor_runner
executable_text: 354304 bytes
executable_data: 133555848 bytes
executable_bss:  419184 bytes


++ '[' false = false ']'
++ backends/arm/scripts/run_fvp.sh --elf=/home/ubuntu/executorch/mv2_u85/mv2_arm_delegate_ethos-u85-128/cmake-out/arm_executor_runner --target=ethos-u85-128


--------------------------------------------------------------------------------
Running /home/ubuntu/executorch/mv2_u85/mv2_arm_delegate_ethos-u85-128/cmake-out/arm_executor_runner for ethos-u85-128 run with FVP:FVP_Corstone_SSE-320 num_macs:128 timeout:600
--------------------------------------------------------------------------------
telnetterminal2: Listening for serial connection on port 5000
telnetterminal5: Listening for serial connection on port 5001
telnetterminal0: Listening for serial connection on port 5002
telnetterminal1: Listening for serial connection on port 5003
I [executorch:arm_executor_runner.cpp:1274 main()] PTE @ 0x74000000 [----ET12]
I [executorch:arm_executor_runner.cpp:602 runner_init()] PTE Model data loaded. Size: 3470624 bytes.
I [executorch:arm_executor_runner.cpp:618 runner_init()] Model buffer loaded, has 1 methods
I [executorch:arm_executor_runner.cpp:628 runner_init()] Running method forward
I [executorch:arm_executor_runner.cpp:639 runner_init()] Set

++ set +x


above is the command for running our other model we built "Mobile net v2" on the ethos u 85 - You will see a a series of commands and potentially strange looking prompts. just wait until you see some stats appear. 

Please note that this command does a variety of things such as calling the "ahead of time" or aot compiler with a variety of flags that tell you what will happen to the model. the one we show in this lab has been set up in how the fvp would want it but we are going to use the complete arm workflow. 

this program will delegate the model to fit the ethos u work flow, it will quantize it as well. our target is simply the board and the model_name specifies what model from the executorch repo we are going to analyse. 

In [19]:
# ---- Ethos-U PT2E quant + lower flow (MobileNetV2) ----
# Drop-in single Jupyter cell adapted from ethos_u_minimal_example.ipynb
#
# Tested assumptions vs your env:
# - executorch.backends.arm.quantizer exports EthosUQuantizer + get_symmetric_quantization_config
# - torchao.pt2e prepare_pt2e/convert_pt2e available
# - Key fix vs your failures: export -> exported_program.module(check_guards=False) to avoid _guards_fn call_module
# - Extra robustness: rewrite any call_function[target=torch.conv2d] -> torch.ops.aten.conv2d.default

import os
import torch
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

from executorch.backends.arm.ethosu import EthosUCompileSpec, EthosUPartitioner
from executorch.backends.arm.quantizer import EthosUQuantizer, get_symmetric_quantization_config
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e

from executorch.exir import EdgeCompileConfig, ExecutorchBackendConfig, to_edge_transform_and_lower
from executorch.extension.export_util.utils import save_pte_program


# ----------------------------
# 0) User-tunable parameters
# ----------------------------
SAVE_DIR = "./ethosu_out"
os.makedirs(SAVE_DIR, exist_ok=True)

PTE_OUT = os.path.join(SAVE_DIR, "mobilenetv2_ethosu.pte")

# Pick your target (edit to match your board / vela config)
# Common examples: "ethos-u55-256", "ethos-u85-256"
ETHOSU_TARGET = "ethos-u55-256"

# These are Vela flags; keep if you want raw output + extra debug.
# You can remove "--debug-force-regor" if you don't want it.
EXTRA_VELA_FLAGS = ["--output-format=raw", "--debug-force-regor"]

# MobileNet input shape
BATCH = 1
H, W = 224, 224


# ----------------------------
# 1) Small helpers
# ----------------------------
def _rewrite_torch_conv2d_to_aten(gm: torch.fx.GraphModule) -> torch.fx.GraphModule:
    """
    Arm quantizer pass pipeline may choke on call_function[target=torch.conv2d]
    because it's a Python builtin, not an ATen op. Rewrite to aten.conv2d.default.
    """
    changed = False
    for n in gm.graph.nodes:
        if n.op == "call_function" and n.target is torch.conv2d:
            n.target = torch.ops.aten.conv2d.default
            changed = True
        # Some graphs may contain F.conv2d (rare here, but cheap to handle)
        if n.op == "call_function" and getattr(n.target, "__name__", None) == "conv2d":
            # Only rewrite if it's torch.nn.functional.conv2d
            mod = getattr(n.target, "__module__", "")
            if mod in ("torch.nn.functional", "torch.nn.modules.utils", "torch.nn.functional.py"):
                n.target = torch.ops.aten.conv2d.default
                changed = True

    if changed:
        gm.graph.lint()
        gm.recompile()
    return gm


def export_stateful_gm(model: torch.nn.Module, example_inputs: tuple[torch.Tensor, ...]) -> torch.fx.GraphModule:
    """
    Export and return a stateful fx.GraphModule without guards.
    This avoids the `_guards_fn` call_module issue you hit.
    """
    model = model.eval()
    exported_program = torch.export.export(model, example_inputs)
    gm = exported_program.module(check_guards=False)  # IMPORTANT
    gm = _rewrite_torch_conv2d_to_aten(gm)
    return gm


def quantize_pt2e_ethosu(
    graph_module: torch.fx.GraphModule,
    example_inputs: tuple[torch.Tensor, ...],
    compile_spec: EthosUCompileSpec,
) -> torch.export.ExportedProgram:
    """
    PT2E post-training quantization using Arm Ethos-U quantizer, then re-export.
    """
    # Configure quantizer
    quantizer = EthosUQuantizer(compile_spec)
    operator_config = get_symmetric_quantization_config()
    quantizer.set_global(operator_config)

    # Prepare (annotate/insert observers), calibrate, convert
    q_gm = prepare_pt2e(graph_module, quantizer)
    q_gm(*example_inputs)  # calibration run
    q_gm = convert_pt2e(q_gm)

    # Re-export quantized module
    q_ep = torch.export.export(q_gm, example_inputs)
    return q_ep


def lower_to_ethosu_and_save(
    exported_program: torch.export.ExportedProgram,
    compile_spec: EthosUCompileSpec,
    out_pte_path: str,
):
    """
    Partition + lower to Ethos-U delegate, convert to ExecuTorch program, save .pte.
    """
    partitioner = EthosUPartitioner(compile_spec)

    edge_pm = to_edge_transform_and_lower(
        exported_program,
        partitioner=[partitioner],
        compile_config=EdgeCompileConfig(_check_ir_validity=False),
    )

    et_pm = edge_pm.to_executorch(
        config=ExecutorchBackendConfig(extract_delegate_segments=False)
    )

    save_pte_program(et_pm, out_pte_path)
    return et_pm


# ----------------------------
# 2) Build model + inputs
# ----------------------------
print("[1/4] Build MobileNetV2 + example inputs")

try:
    weights = MobileNet_V2_Weights.DEFAULT
    baseline = mobilenet_v2(weights=weights).eval()
    print("Loaded MobileNetV2 weights:", weights)
except Exception as e:
    baseline = mobilenet_v2(weights=None).eval()
    print("WARNING: Could not load pretrained weights (falling back to weights=None). Error:", repr(e))

example_inputs = (torch.randn(BATCH, 3, H, W),)


# ----------------------------
# 3) Export (guard-free) -> Quantize -> Re-export
# ----------------------------
print("\n[2/4] Export to stateful GraphModule (check_guards=False) ...")
gm = export_stateful_gm(baseline, example_inputs)
print("Exported graph module type:", type(gm))
# Uncomment if you want to inspect:
# _ = gm.print_readable()

print("\n[3/4] Create Ethos-U compile spec + PT2E quantize ...")
compile_spec = EthosUCompileSpec(
    target=ETHOSU_TARGET,
    # Adjust to your platform; these are the same style as the minimal example
    system_config="Ethos_U85_SYS_DRAM_Mid",
    memory_mode="Shared_Sram",
    extra_flags=list(EXTRA_VELA_FLAGS),
)

q_exported_program = quantize_pt2e_ethosu(gm, example_inputs, compile_spec)
print("Quantized ExportedProgram ready.")


# ----------------------------
# 4) Lower to Ethos-U + save .pte
# ----------------------------
print("\n[4/4] Lower to Ethos-U delegate + save .pte ...")
et_pm = lower_to_ethosu_and_save(q_exported_program, compile_spec, PTE_OUT)

print("\nDONE ✅")
print("Saved:", PTE_OUT)

# Optional: inspect final lowered graph
# _ = et_pm.exported_program().module(check_guards=False).print_readable()

[1/4] Build MobileNetV2 + example inputs
Loaded MobileNetV2 weights: MobileNet_V2_Weights.IMAGENET1K_V2

[2/4] Export to stateful GraphModule (check_guards=False) ...
Exported graph module type: <class 'torch.fx.graph_module.GraphModule.__new__.<locals>.GraphModuleImpl'>

[3/4] Create Ethos-U compile spec + PT2E quantize ...


/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/backends/arm/quantizer/quantization_config.py:171: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(act_scale * weight_scale).to(


Quantized ExportedProgram ready.

[4/4] Lower to Ethos-U delegate + save .pte ...


/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)



Network summary for out
Accelerator configuration               Ethos_U55_256
System configuration             Ethos_U85_SYS_DRAM_Mid
Memory mode                               Shared_Sram
Accelerator clock                                1000 MHz
Design peak SRAM bandwidth                      14.90 GB/s
Design peak DRAM bandwidth                       5.59 GB/s

Total SRAM used                               1474.20 KiB
Total DRAM used                               3270.50 KiB

CPU operators = 0 (0.0%)
NPU operators = 117 (100.0%)

Average SRAM bandwidth                           3.55 GB/s
Input   SRAM bandwidth                          14.19 MB/batch
Weight  SRAM bandwidth                           8.70 MB/batch
Output  SRAM bandwidth                           8.96 MB/batch
Total   SRAM bandwidth                          32.61 MB/batch
Total   SRAM bandwidth            per input     32.61 MB/inference (batch size 1)

Average DRAM bandwidth                           0.35 GB/s
Input   D

/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)



DONE ✅
Saved: ./ethosu_out/mobilenetv2_ethosu.pte


In [20]:
from executorch.devtools.backend_debug import get_delegation_info

gm = et_pm.exported_program().graph_module
info = get_delegation_info(gm)

print(info.get_summary())
print(info.get_operator_delegation_dataframe())

Total delegated subgraphs: 1
Number of delegated nodes: 349
Number of non-delegated nodes: 5

                                              op_type  \
0                                               alloc   
1                                     aten_add_tensor   
2                            aten_convolution_default   
3                               aten_hardtanh_default   
4                                 aten_linear_default   
5                                       aten_mean_dim   
6                              aten_view_copy_default   
7              dim_order_ops__clone_dim_order_default   
8                                             getitem   
9   quantized_decomposed_dequantize_per_channel_de...   
10  quantized_decomposed_dequantize_per_tensor_def...   
11   quantized_decomposed_quantize_per_tensor_default   
12                                              Total   

    occurrences_in_delegated_graphs  occurrences_in_non_delegated_graphs  
0                               

In [24]:
# -------------------------
# LRN flow (drop-in cell)
# -------------------------
# Assumes the previous (working) cell already defined:
#   - compile_spec
#   - example_inputs
#   - SAVE_DIR
#   - lower_to_ethosu_and_save(...)   (your helper)
#
# This cell avoids the `_guards_fn` / `call_module` failure by exporting and using
# `exported_program.module(check_guards=False)` exactly like the known-good minimal example.

import os
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e

# ExecuTorch / Arm quantizer imports (match the minimal example pattern)
from executorch.backends.arm.quantizer.arm_quantizer import EthosUQuantizer
from executorch.backends.arm.quantizer import get_symmetric_quantization_config


class MobileNetV2WithLRN(nn.Module):
    """
    Inserts an LRN op in the forward path.
    Adjust placement if you need it elsewhere; this version applies LRN
    right after feature extraction and before global pooling/classifier.
    """
    def __init__(self, weights=MobileNet_V2_Weights.DEFAULT, lrn_size=5, alpha=1e-4, beta=0.75, k=1.0):
        super().__init__()
        m = mobilenet_v2(weights=weights).eval()
        self.features = m.features
        self.avgpool = m.avgpool
        self.classifier = m.classifier
        self.lrn = nn.LocalResponseNorm(size=lrn_size, alpha=alpha, beta=beta, k=k)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.lrn(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


def export_to_stateful_gm_no_guards(model: nn.Module, example_inputs):
    """
    Export then return a *stateful* graph module with params as get_attr nodes,
    and with guards disabled to avoid `_guards_fn` / call_module issues.
    """
    model = model.eval()
    ep = torch.export.export(model, example_inputs)
    gm = ep.module(check_guards=False)  # key line (matches minimal example)
    return gm


def quantize_stateful_gm_for_ethosu(stateful_gm: torch.nn.Module, example_inputs, compile_spec):
    """
    PT2E quantize a *stateful* exported GraphModule using the Ethos-U quantizer.
    Returns an ExportedProgram of the quantized module.
    """
    # Create quantizer (compile_spec must be the list-style compile_spec you used for baseline)
    quantizer = EthosUQuantizer(compile_spec)

    # Use symmetric config (matches minimal example); keep per-tensor as in your earlier flow.
    quantizer.set_global(get_symmetric_quantization_config(is_per_channel=False))

    # Prepare (annotate + insert observers)
    q_gm = prepare_pt2e(stateful_gm, quantizer)

    # Calibration run
    with torch.no_grad():
        _ = q_gm(*example_inputs)

    # Convert (swap in quant/dequant + quantized ops)
    q_gm = convert_pt2e(q_gm)

    # Re-export the quantized graph module to an ExportedProgram (clean handoff to lowering)
    q_ep = torch.export.export(q_gm, example_inputs)
    return q_ep


# ---- 1) Build LRN model
print("\n[LRN 0/3] Building MobileNetV2 + LRN model ...")
lrn_model = MobileNetV2WithLRN().eval()

# ---- 2) Export to a stateful GM with guards disabled (prevents `_guards_fn` node)
print("[LRN 1/3] Exporting (no guards) ...")
lrn_stateful_gm = export_to_stateful_gm_no_guards(lrn_model, example_inputs)

# Optional: quick sanity print
# _ = lrn_stateful_gm.print_readable()

# ---- 3) Quantize + Lower + Save using your existing helper(s)
print("[LRN 2/3] Quantizing (PT2E + Ethos-U quantizer) ...")
lrn_quant_ep = quantize_stateful_gm_for_ethosu(lrn_stateful_gm, example_inputs, compile_spec)

print("[LRN 3/3] Lowering to Ethos-U and saving .pte ...")
lrn_pte_path = os.path.join(SAVE_DIR, "mobilenetv2_lrn_ethosu.pte")

# Your helper name/signature may differ; try the most common pattern first.
try:
    lrn_edge_pm = lower_to_ethosu_and_save(lrn_quant_ep, compile_spec, lrn_pte_path)
except TypeError:
    # Fallback: some helpers take a module/graphmodule instead of ExportedProgram
    lrn_edge_pm = lower_to_ethosu_and_save(lrn_stateful_gm, compile_spec, lrn_pte_path)

print("Saved:", lrn_pte_path)


[LRN 0/3] Building MobileNetV2 + LRN model ...


AttributeError: 'MobileNetV2' object has no attribute 'avgpool'

In [25]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

# -------------------------
# Models
# -------------------------
class MobileNetV2_LRN_Features(nn.Module):
    """
    MobileNetV2 with an LRN inserted *after* features and before global pooling.
    Does NOT rely on a .avgpool attribute (it doesn't exist on MobileNetV2).
    """
    def __init__(self, weights=MobileNet_V2_Weights.DEFAULT,
                 lrn_size=5, alpha=1e-4, beta=0.75, k=1.0):
        super().__init__()
        base = mobilenet_v2(weights=weights).eval()
        self.features = base.features
        self.classifier = base.classifier
        self.lrn = nn.LocalResponseNorm(size=lrn_size, alpha=alpha, beta=beta, k=k)

    def forward(self, x):
        x = self.features(x)
        x = self.lrn(x)
        # mimic torchvision MobileNetV2 forward: global avg pool then flatten then classifier
        x = F.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


def export_ep_no_guards(model: nn.Module, example_inputs):
    model = model.eval()
    ep = torch.export.export(model, example_inputs)
    # critical: avoid guards -> avoids `_guards_fn` call_module failures downstream
    gm = ep.module(check_guards=False)
    # Re-export the stateful module so downstream expects an ExportedProgram
    return torch.export.export(gm, example_inputs)


# -------------------------
# 1) Baseline non-quantized
# -------------------------
print("\n[BASELINE/FLOAT 1/3] Build model")
baseline = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()

print("[BASELINE/FLOAT 2/3] Export (no guards)")
baseline_float_ep = export_ep_no_guards(baseline, example_inputs)

print("[BASELINE/FLOAT 3/3] Lower + save")
baseline_float_pte = os.path.join(SAVE_DIR, "mobilenetv2_float_ethosu.pte")
baseline_float_edge_pm = lower_to_ethosu_and_save(baseline_float_ep, compile_spec, baseline_float_pte)
print("Saved:", baseline_float_pte)

# -------------------------
# 2) LRN non-quantized
# -------------------------
print("\n[LRN/FLOAT 1/3] Build model")
lrn_float = MobileNetV2_LRN_Features().eval()

print("[LRN/FLOAT 2/3] Export (no guards)")
lrn_float_ep = export_ep_no_guards(lrn_float, example_inputs)

print("[LRN/FLOAT 3/3] Lower + save")
lrn_float_pte = os.path.join(SAVE_DIR, "mobilenetv2_lrn_float_ethosu.pte")
lrn_float_edge_pm = lower_to_ethosu_and_save(lrn_float_ep, compile_spec, lrn_float_pte)
print("Saved:", lrn_float_pte)

print("\nDone. You can now open both .pte files in Model Explorer and compare delegation.")


[BASELINE/FLOAT 1/3] Build model
[BASELINE/FLOAT 2/3] Export (no guards)
[BASELINE/FLOAT 3/3] Lower + save


/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size

Saved: ./ethosu_out/mobilenetv2_float_ethosu.pte

[LRN/FLOAT 1/3] Build model
[LRN/FLOAT 2/3] Export (no guards)
[LRN/FLOAT 3/3] Lower + save


/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size

Saved: ./ethosu_out/mobilenetv2_lrn_float_ethosu.pte

Done. You can now open both .pte files in Model Explorer and compare delegation.


In [27]:
# ============================================================
# Delegation comparison: baseline (float) vs LRN (float)
# Assumes previous cell created:
#   baseline_float_edge_pm
#   lrn_float_edge_pm
# ============================================================

from collections import Counter

def _try_get_gm(edge_pm):
    """
    Best-effort extraction of a torch.fx.GraphModule for inspection.
    Works across common ExecuTorch/EXIR shapes.
    """
    # Most common: EdgeProgramManager has exported_program()
    if hasattr(edge_pm, "exported_program") and callable(edge_pm.exported_program):
        ep = edge_pm.exported_program()
        if hasattr(ep, "module"):
            # Avoid guards if supported
            try:
                return ep.module(check_guards=False)
            except TypeError:
                return ep.module()

    # Sometimes stored directly
    if hasattr(edge_pm, "graph_module"):
        return edge_pm.graph_module

    # Some builds expose a "module()" accessor
    if hasattr(edge_pm, "module") and callable(edge_pm.module):
        return edge_pm.module()

    raise RuntimeError("Could not extract a GraphModule from edge_pm (API mismatch).")

def _is_delegated_call(node):
    """
    Heuristic: delegated regions usually show up as call_function/call_method to a delegate call.
    We match substrings (nightly-safe) rather than a single symbol.
    """
    if node.op not in ("call_function", "call_method"):
        return False
    t = str(node.target)
    patterns = (
        "call_delegate",
        "delegate_call",
        "backend_call",
        "executorch_call_delegate",
        "lowered_module",
    )
    return any(p in t for p in patterns)

def _op_name(node):
    if node.op == "call_function":
        return getattr(node.target, "__name__", str(node.target))
    if node.op == "call_method":
        return f"method::{node.target}"
    if node.op == "call_module":
        return f"module::{node.target}"
    return node.op

def summarize_delegation(edge_pm, name, top_k=20):
    gm = _try_get_gm(edge_pm)

    total = 0
    delegated = 0
    nondeleg_ops = Counter()

    for n in gm.graph.nodes:
        if n.op in ("placeholder", "output"):
            continue
        total += 1
        if _is_delegated_call(n):
            delegated += 1
        else:
            nondeleg_ops[_op_name(n)] += 1

    nondeleg = total - delegated
    pct = (100.0 * delegated / total) if total else 0.0
    return {
        "name": name,
        "total": total,
        "delegated": delegated,
        "nondeleg": nondeleg,
        "pct": pct,
        "top_nondeleg": nondeleg_ops.most_common(top_k),
    }

# ---- Run summaries
results = []
results.append(summarize_delegation(baseline_float_edge_pm, "MobileNetV2 (float, baseline)"))
results.append(summarize_delegation(lrn_float_edge_pm, "MobileNetV2 + LRN (float)"))

# ---- Print comparison table
hdr = f"{'Model':<34} {'Nodes':>8} {'Delegated':>10} {'Non-deleg':>10} {'% delegated':>12}"
print("\n" + hdr)
print("-" * len(hdr))
for r in results:
    print(f"{r['name']:<34} {r['total']:>8} {r['delegated']:>10} {r['nondeleg']:>10} {r['pct']:>11.1f}%")

# ---- Print top non-delegated ops (useful to spot LRN fallout)
for r in results:
    print(f"\nTop non-delegated ops for: {r['name']}")
    if not r["top_nondeleg"]:
        print("  (none)")
        continue
    for op, cnt in r["top_nondeleg"]:
        print(f"  {op:<45} {cnt}")


Model                                 Nodes  Delegated  Non-deleg  % delegated
------------------------------------------------------------------------------
MobileNetV2 (float, baseline)           777          0        777         0.0%
MobileNetV2 + LRN (float)               797          0        797         0.0%

Top non-delegated ops for: MobileNetV2 (float, baseline)
  get_attr                                      314
  alloc                                         257
  convolution.out                               52
  _native_batch_norm_legit_no_training.out      52
  getitem                                       52
  hardtanh.out                                  35
  add.out                                       10
  mean.out                                      1
  view                                          1
  _clone_dim_order.out                          1
  permute_copy.out                              1
  addmm.out                                     1

Top non-delegat

In [41]:
import os
import collections
import torch
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

# ExecuTorch / Arm / PT2E quant
from executorch.backends.arm.ethosu import EthosUCompileSpec, EthosUPartitioner
from executorch.backends.arm.quantizer import EthosUQuantizer, get_symmetric_quantization_config
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from executorch.exir import EdgeCompileConfig, ExecutorchBackendConfig, to_edge_transform_and_lower

# -------------------------
# Config
# -------------------------
SAVE_DIR = os.environ.get("SAVE_DIR", "./out")
os.makedirs(SAVE_DIR, exist_ok=True)

example_inputs = (torch.randn(1, 3, 224, 224),)

compile_spec = EthosUCompileSpec(
    target="ethos-u85-256",
    system_config="Ethos_U85_SYS_DRAM_Mid",
    memory_mode="Shared_Sram",
    extra_flags=["--output-format=raw"],
)

# -------------------------
# Helpers
# -------------------------
def export_ep_no_guards(model: torch.nn.Module, example_inputs):
    model = model.eval()
    ep = torch.export.export(model, example_inputs)
    gm = ep.module(check_guards=False)               # avoid `_guards_fn`
    return torch.export.export(gm, example_inputs)   # re-export stateful module

def lower_to_ethosu_and_save(ep, compile_spec, pte_path: str):
    edge_pm = to_edge_transform_and_lower(
        ep,
        compile_config=EdgeCompileConfig(_check_ir_validity=False),
        partitioner=[EthosUPartitioner(compile_spec)],
    )
    et_pm = edge_pm.to_executorch(ExecutorchBackendConfig(extract_delegate_segments=True))
    et_pm.save(pte_path)
    return edge_pm

def _get_graph_module_from_edge_pm(edge_pm):
    # Best-effort across ExecuTorch variants
    if hasattr(edge_pm, "exported_program"):
        ep = edge_pm.exported_program()
        if hasattr(ep, "graph_module"):
            return ep.graph_module
    for attr in ["graph_module", "gm", "_graph_module"]:
        if hasattr(edge_pm, attr):
            return getattr(edge_pm, attr)
    raise RuntimeError("Couldn't locate a graph_module on edge_pm (API mismatch).")

def delegation_stats(edge_pm, name):
    gm = _get_graph_module_from_edge_pm(edge_pm)
    nodes = list(gm.graph.nodes)

    delegate_nodes = [
        n for n in nodes
        if n.op == "call_function"
        and ("call_delegate" in str(n.target)
             or "executorch_call_delegate" in str(n.target))
    ]

    print(f"\n{name}")
    print(f"  Top-level FX nodes: {len(nodes)}")
    print(f"  Delegate call nodes: {len(delegate_nodes)}")

    if len(delegate_nodes) == 0:
        print("  ➜ No Ethos-U delegation detected (float graph runs on host).")
    else:
        print("  ➜ Ethos-U delegate present.")
        print("    The heavy compute (convs, etc.) is encapsulated inside this delegate segment.")
        print("    Wrapper ops (quant/dequant/alloc) remain on host.")

In [42]:
print("[CELL 1] Float MobileNetV2 -> lower (expect ~0 delegation; Ethos-U targets int8)")

model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()
ep_float = export_ep_no_guards(model, example_inputs)

pte_float = os.path.join(SAVE_DIR, "mobilenetv2_float_ethosu.pte")
edge_pm_float = lower_to_ethosu_and_save(ep_float, compile_spec, pte_float)
print("Saved:", pte_float)

delegation_stats(edge_pm_float, "MobileNetV2 (float)")

[CELL 1] Float MobileNetV2 -> lower (expect ~0 delegation; Ethos-U targets int8)


/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size

Saved: ./out/mobilenetv2_float_ethosu.pte

MobileNetV2 (float)
  Top-level FX nodes: 779
  Delegate call nodes: 0
  ➜ No Ethos-U delegation detected (float graph runs on host).


In [43]:
print("[CELL 2] PT2E int8 quant MobileNetV2 -> lower (expect delegation now)")

model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()

# Export -> stateful GM without guards (critical)
ep = torch.export.export(model, example_inputs)
gm = ep.module(check_guards=False)

# Quantizer (Ethos-U) + symmetric config
quantizer = EthosUQuantizer(compile_spec)
quantizer.set_global(get_symmetric_quantization_config())

# Prepare (annotate) -> calibrate -> convert
q_gm = prepare_pt2e(gm, quantizer)
with torch.no_grad():
    q_gm(*example_inputs)  # calibration pass
q_gm = convert_pt2e(q_gm)

# Re-export quantized GM as ExportedProgram for lowering
ep_int8 = torch.export.export(q_gm, example_inputs)

pte_int8 = os.path.join(SAVE_DIR, "mobilenetv2_int8_ethosu.pte")
edge_pm_int8 = lower_to_ethosu_and_save(ep_int8, compile_spec, pte_int8)
print("Saved:", pte_int8)



[CELL 2] PT2E int8 quant MobileNetV2 -> lower (expect delegation now)


/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/backends/arm/quantizer/quantization_config.py:171: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(act_scale * weight_scale).to(
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(tr


Network summary for out
Accelerator configuration               Ethos_U85_256
System configuration             Ethos_U85_SYS_DRAM_Mid
Memory mode                               Shared_Sram
Accelerator clock                                1000 MHz
Design peak SRAM bandwidth                      29.80 GB/s
Design peak DRAM bandwidth                      11.18 GB/s

Total SRAM used                               1474.58 KiB
Total DRAM used                               3362.17 KiB

CPU operators = 0 (0.0%)
NPU operators = 66 (100.0%)

Average SRAM bandwidth                           5.64 GB/s
Input   SRAM bandwidth                          16.07 MB/batch
Weight  SRAM bandwidth                          11.20 MB/batch
Output  SRAM bandwidth                           8.19 MB/batch
Total   SRAM bandwidth                          36.58 MB/batch
Total   SRAM bandwidth            per input     36.58 MB/inference (batch size 1)

Average DRAM bandwidth                           0.51 GB/s
Input   DR

/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)


In [45]:
delegation_stats(edge_pm_int8, "MobileNetV2 (int8 PT2E)")


MobileNetV2 (int8 PT2E)
  Top-level FX nodes: 9
  Delegate call nodes: 1
  ➜ Ethos-U delegate present.
    The heavy compute (convs, etc.) is encapsulated inside this delegate segment.
    Wrapper ops (quant/dequant/alloc) remain on host.


In [ ]:
# === INT8 + LRN variant (non-TOSA-friendly) ===

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

class MobileNetV2_LRN(nn.Module):
    def __init__(self):
        super().__init__()
        base = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()
        self.features = base.features
        self.classifier = base.classifier
        self.lrn = nn.LocalResponseNorm(5)

    def forward(self, x):
        x = self.features(x)
        x = self.lrn(x)                    # <- breaks TOSA compatibility
        x = F.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


print("\n=== INT8 + LRN MODEL ===")

# Build model
lrn_model = MobileNetV2_LRN().eval()

# Export (no guards)
ep = torch.export.export(lrn_model, example_inputs)
gm = ep.module(check_guards=False)

# Quantize (reuse your PT2E flow)
quantizer = EthosUQuantizer(compile_spec)
quantizer.set_global(get_symmetric_quantization_config())

q_gm = prepare_pt2e(gm, quantizer)
with torch.no_grad():
    q_gm(*example_inputs)
q_gm = convert_pt2e(q_gm)

ep_lrn_int8 = torch.export.export(q_gm, example_inputs)

# Lower
pte_lrn_int8 = os.path.join(SAVE_DIR, "mobilenetv2_lrn_int8_ethosu.pte")
edge_pm_lrn_int8 = lower_to_ethosu_and_save(ep_lrn_int8, compile_spec, pte_lrn_int8)

print("Saved:", pte_lrn_int8)

# Compare delegation
delegation_stats(edge_pm_lrn_int8, "MobileNetV2 + LRN (int8)")